In [ ]:
#@title Ячейка 0 — Drive + автосборка doc_map.py
from google.colab import drive
drive.mount('/content/drive')
import os, glob

BASE = "/content/drive/MyDrive/rag_exp"
assert os.path.exists(BASE), f"Нет папки {BASE}"

config = '''# -*- coding: utf-8 -*-
import os, glob
CORPUS = "/content/drive/MyDrive/rag_exp/corpus_phase1"
NAME_RULES = {
    "РД 50-726": "rd_50_726_92",
    "2.701": "gost_2_701",
    "30893": "gost_30893_1",
    "2-020101-174": "nd_2_020101_174_ch2",
    "2-09-006": "nd_2_09_006_kn6",
    "Часть VIII": "pkps_chast_8",
    "Часть XI": "pkps_chast_11",
    "4110": "ost_5r_4110",
}
FILE_TO_ID = {}
for path in glob.glob(f"{CORPUS}/*.pdf"):
    fname = os.path.basename(path)
    for key, doc_id in NAME_RULES.items():
        if key in fname:
            FILE_TO_ID[fname] = doc_id
            break
'''
with open(f"{BASE}/doc_map.py", "w", encoding="utf-8") as f:
    f.write(config)
print("doc_map.py пересоздан")



Mounted at /content/drive
doc_map.py пересоздан


In [ ]:
#@title Ячейка 1 — загрузка карты документов (FILE_TO_ID)
import sys, importlib
sys.path.append("/content/drive/MyDrive/rag_exp")
import doc_map; importlib.reload(doc_map)
from doc_map import FILE_TO_ID
print("Загружено документов:", len(FILE_TO_ID))
for f, d in FILE_TO_ID.items():
    print(f"  {d:22s} <- {f}")




Загружено документов: 7
  pkps_chast_11          <- ПКПС Часть XI (Электрическое оборудование, изд.2017).pdf
  pkps_chast_8           <- ПКПС Часть VIII (Системы и трубопроводы, изд.2018).pdf
  rd_50_726_92           <- РД 50-726-92.pdf
  gost_2_701             <- ГОСТ 2.701-2008.pdf
  gost_30893_1           <- ГОСТ 30893.1-2002.pdf
  nd_2_09_006_kn6        <- НД №2-09-006 кн.6 (переиздан как 2-039901-005, 2018).pdf
  nd_2_020101_174_ch2    <- НД №2-020101-174 ч.2 (Правила РС, корпус).pdf


In [ ]:
#@title Ячейка 2 — установка зависимостей
!pip install -q -U "transformers>=4.51.0" "sentence-transformers>=2.7.0" bitsandbytes accelerate
!pip install -q scikit-learn pandas pyyaml rank_bm25 openai pymupdf
print("deps ok")



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 159.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 44.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 34.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 112.3 MB/s eta 0:00:00
deps ok


In [ ]:
#@title Ячейка 3 — BASE, HF_HOME, папки
import os
BASE = "/content/drive/MyDrive/rag_exp"
os.makedirs(f"{BASE}/index_cache", exist_ok=True)
os.environ.pop("HF_HOME", None)
os.environ["HF_HOME"] = "/content/hf_cache"
os.makedirs("/content/hf_cache", exist_ok=True)
print("BASE =", BASE, "| HF_HOME =", os.environ["HF_HOME"])



BASE = /content/drive/MyDrive/rag_exp | HF_HOME = /content/hf_cache


In [ ]:
#@title Ячейка 4 — проверка GPU
import torch
assert torch.cuda.is_available(), "GPU не подключён! Среда выполнения → Сменить → T4 GPU"
print(torch.cuda.get_device_name(0),
      f"| VRAM {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")


NVIDIA L4 | VRAM 23.7 GB


In [ ]:
#@title Ячейка 5 — загрузка модели Qwen3-Embedding-4B (int8)
from transformers import AutoModel, AutoTokenizer, BitsAndBytesConfig
MODEL = "Qwen/Qwen3-Embedding-4B"
tokenizer = AutoTokenizer.from_pretrained(MODEL, padding_side="left")
bnb = BitsAndBytesConfig(load_in_8bit=True)
model = AutoModel.from_pretrained(MODEL, quantization_config=bnb, device_map="auto")
model.eval()
print(f"VRAM после загрузки: {torch.cuda.memory_allocated()/1e9:.1f} GB")
print("model:", type(model), "| tokenizer:", type(tokenizer))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.26k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/30.4k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

VRAM после загрузки: 4.4 GB
model: <class 'transformers.models.qwen3.modeling_qwen3.Qwen3Model'> | tokenizer: <class 'transformers.models.qwen2.tokenization_qwen2.Qwen2Tokenizer'>


In [ ]:
#@title Ячейка 6 — импорт rag_common + init_common
import sys, importlib
sys.path.append(BASE)
importlib.invalidate_caches()
import rag_common; importlib.reload(rag_common)
from rag_common import init_common, build_or_load_index, SparseIndex, run_grid, compare_strategies
init_common(BASE, MODEL, model, tokenizer)
print("rag_common ok")

rag_common init: MODEL=Qwen/Qwen3-Embedding-4B, INDEX_CACHE=/content/drive/MyDrive/rag_exp/index_cache, reranker=—, oai=—
rag_common ok


In [ ]:
#@title Ячейка 7 — чтение всех PDF корпуса → real_docs
import fitz, os
BASE = "/content/drive/MyDrive/rag_exp"          # страховка от None
CORPUS_DIR = f"{BASE}/corpus_phase1"
assert os.path.isdir(CORPUS_DIR), f"Нет папки {CORPUS_DIR}"
real_docs = []
for fname, doc_id in FILE_TO_ID.items():
    path = os.path.join(CORPUS_DIR, fname)
    assert os.path.exists(path), f"Нет файла: {path}"
    pdf = fitz.open(path)
    text = "\n".join(page.get_text() for page in pdf)
    pdf.close()
    real_docs.append({"id": doc_id, "text": text})
    print(f"{doc_id:22s} <- {fname[:45]:45s} ({len(text)} символов)")
print(f"\nДокументов: {len(real_docs)}")

pkps_chast_11          <- ПКПС Часть XI (Электрическое оборудование, из (517249 символов)
pkps_chast_8           <- ПКПС Часть VIII (Системы и трубопроводы, изд. (462765 символов)
rd_50_726_92           <- РД 50-726-92.pdf                              (130509 символов)
gost_2_701             <- ГОСТ 2.701-2008.pdf                           (41739 символов)
gost_30893_1           <- ГОСТ 30893.1-2002.pdf                         (17713 символов)
nd_2_09_006_kn6        <- НД №2-09-006 кн.6 (переиздан как 2-039901-005 (375828 символов)
nd_2_020101_174_ch2    <- НД №2-020101-174 ч.2 (Правила РС, корпус).pdf (785091 символов)

Документов: 7


In [ ]:
#@title Ячейка 7.5 — привязка rag_common + патч encode
import rag_common, torch, time
import torch.nn.functional as F

init_common("/content/drive/MyDrive/rag_exp", "Qwen/Qwen3-Embedding-4B", model, tokenizer)
rag_common.model = model
rag_common.tokenizer = tokenizer
rag_common.MODEL = "Qwen/Qwen3-Embedding-4B"

@torch.no_grad()
def encode_batched(texts, dim=None, max_length=512, batch_size=16):
    out = []
    total = len(texts)
    print(f"  всего текстов на эмбеддинг: {total}")
    t0 = time.time()
    for i in range(0, total, batch_size):
        batch = texts[i:i + batch_size]
        enc = rag_common.tokenizer(batch, padding=True, truncation=True,
                                   max_length=max_length, return_tensors="pt").to(rag_common.model.device)
        hidden = rag_common.model(**enc).last_hidden_state
        vecs = rag_common.last_token_pool(hidden, enc["attention_mask"])
        if dim:
            vecs = vecs[:, :dim]
        vecs = F.normalize(vecs, p=2, dim=1)
        out.append(vecs.float().cpu())
        del enc, hidden, vecs
        torch.cuda.empty_cache()
        done = i + len(batch)
        if (i // batch_size) % 10 == 0:
            print(f"  эмбеддинг: {done}/{total}  ({time.time()-t0:.0f} сек)")
    print(f"  готово: {total} за {time.time()-t0:.0f} сек")
    return torch.cat(out, dim=0)

rag_common.encode = encode_batched
print("encode:", rag_common.encode.__name__,
      "| MODEL:", rag_common.MODEL,
      "| model:", type(rag_common.model).__name__,
      "| tokenizer:", type(rag_common.tokenizer).__name__)

rag_common init: MODEL=Qwen/Qwen3-Embedding-4B, INDEX_CACHE=/content/drive/MyDrive/rag_exp/index_cache, reranker=—, oai=—
encode: encode_batched | MODEL: Qwen/Qwen3-Embedding-4B | model: Qwen3Model | tokenizer: Qwen2Tokenizer


In [ ]:
#@title Ячейка 8 — построение боевого индекса (cache)
idx_real = build_or_load_index(real_docs, dim=2048, quant="int8")
print(f"Чанков: {len(idx_real['chunks'])}, форма: {idx_real['vectors'].shape}")
print("index_key:", idx_real["key"])

[cache HIT] индекс e73eed43252c4b6c: 5071 чанков загружено с Drive
Чанков: 5071, форма: (5071, 2048)
index_key: e73eed43252c4b6c
